# Representation Demystifies Flat Minima
## Notebook 23: Correlation vs Causation

Notebook 17 showed:

> Same function. Different sharpness.

This notebook asks the next question:

> If flatness can change under reparameterization, what remains of the claim that flatness causes generalization?

We build a synthetic experiment where sharpness correlates with test error, then show how reparameterization can change sharpness while preserving the model behavior.

## Output files

Running this notebook creates:

- `figures/23_original_correlation.png`
- `figures/23_reparameterized_scatter.png`
- `figures/23_correlation_by_scale.png`
- `figures/23_before_after_correlation.png`
- `results/23_correlation_results.csv`
- `results/23_correlation_summary.json`
- `downloads/23_correlation_vs_causation_outputs.zip`

The final cell displays direct download links in Colab/Jupyter.

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DOWNLOADS = ROOT / "downloads"

for path in [FIGURES, RESULTS, DOWNLOADS]:
    path.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(260505209)

print("ROOT:", ROOT)
print("FIGURES:", FIGURES)
print("RESULTS:", RESULTS)
print("DOWNLOADS:", DOWNLOADS)

## 1. Build a synthetic correlation

We create a toy collection of models.

Each model has:

- a latent complexity value
- a measured sharpness value
- a test error value

The construction intentionally makes sharpness correlate with test error.

This represents the observation:

> Flatter minima often generalize better.

In [ ]:
n_models = 80

complexity = rng.uniform(0.0, 1.0, n_models)

# A synthetic world where complexity drives both sharpness and test error.
base_sharpness = np.exp(2.5 * complexity) + rng.normal(0, 0.25, n_models)
base_sharpness = np.clip(base_sharpness, 0.05, None)

test_error = 0.08 + 0.22 * complexity + rng.normal(0, 0.015, n_models)
test_error = np.clip(test_error, 0.0, None)

train_error = 0.03 + 0.03 * complexity + rng.normal(0, 0.006, n_models)
train_error = np.clip(train_error, 0.0, None)

models = pd.DataFrame({
    "model_id": np.arange(n_models),
    "complexity": complexity,
    "sharpness_original": base_sharpness,
    "train_error": train_error,
    "test_error": test_error,
})

corr_original = models["sharpness_original"].corr(models["test_error"])

print("Original sharpness-test error correlation:", round(corr_original, 3))
models.head()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(models["sharpness_original"], models["test_error"], alpha=0.8)

z = np.polyfit(models["sharpness_original"], models["test_error"], 1)
x_line = np.linspace(models["sharpness_original"].min(), models["sharpness_original"].max(), 200)
y_line = z[0] * x_line + z[1]
plt.plot(x_line, y_line, linestyle="--", label=f"corr={corr_original:.3f}")

plt.xlabel("measured sharpness")
plt.ylabel("test error")
plt.title("Original coordinates: sharpness correlates with test error")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "23_original_correlation.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 2. Reparameterize sharpness without changing behavior

From Notebook 17:

\[
w = su
\]

changes Hessian sharpness by:

\[
H_u = s^2 H_w
\]

while preserving the represented function at the optimum.

Here, each model receives a different reparameterization scale.

The model behavior does not change:

- train error unchanged
- test error unchanged
- predictions conceptually unchanged

But measured sharpness changes.

In [ ]:
# Model-specific reparameterization scales.
# Independent of test error, so this scrambles the sharpness measurement.
scale_s = np.exp(rng.uniform(np.log(0.25), np.log(4.0), n_models))

models["scale_s"] = scale_s
models["sharpness_reparameterized"] = models["sharpness_original"] * (scale_s ** 2)
models["sharpness_ratio"] = models["sharpness_reparameterized"] / models["sharpness_original"]

corr_reparam = models["sharpness_reparameterized"].corr(models["test_error"])

print("Original corr:", round(corr_original, 3))
print("Reparameterized corr:", round(corr_reparam, 3))
print("Sharpness ratio range:", models["sharpness_ratio"].min(), "to", models["sharpness_ratio"].max())

models.head()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    models["sharpness_reparameterized"],
    models["test_error"],
    alpha=0.8
)

z = np.polyfit(models["sharpness_reparameterized"], models["test_error"], 1)
x_line = np.linspace(models["sharpness_reparameterized"].min(), models["sharpness_reparameterized"].max(), 200)
y_line = z[0] * x_line + z[1]
plt.plot(x_line, y_line, linestyle="--", label=f"corr={corr_reparam:.3f}")

plt.xlabel("reparameterized measured sharpness")
plt.ylabel("test error")
plt.title("Reparameterized coordinates: behavior preserved, sharpness changed")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "23_reparameterized_scatter.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 3. Same errors, different measured sharpness

The test error is unchanged for every model.

Only the coordinate-dependent sharpness measurement changes.

This is the causal warning:

> A variable can be predictive in one coordinate system without being a coordinate-invariant explanation.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(models["sharpness_original"], models["test_error"], alpha=0.6, label="original")
plt.scatter(models["sharpness_reparameterized"], models["test_error"], alpha=0.6, label="reparameterized")

plt.xscale("log")
plt.xlabel("measured sharpness")
plt.ylabel("test error")
plt.title("Same test errors, different sharpness coordinates")
plt.legend()
plt.grid(True, which="both")

fig_path = FIGURES / "23_before_after_correlation.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 4. Correlation under global scale changes

If every model receives the same global reparameterization scale, correlation remains the same.

That is important.

The problem is not that every coordinate change destroys every correlation.

The problem is that sharpness is not automatically an invariant property of the function.

In [ ]:
global_scales = np.array([0.25, 0.5, 1, 2, 4, 8])
corr_rows = []

for s in global_scales:
    sharpness_global = models["sharpness_original"] * (s ** 2)
    corr_rows.append({
        "global_scale_s": float(s),
        "correlation": float(sharpness_global.corr(models["test_error"])),
        "median_sharpness": float(np.median(sharpness_global))
    })

corr_by_scale = pd.DataFrame(corr_rows)
corr_by_scale

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(corr_by_scale["global_scale_s"], corr_by_scale["correlation"], marker="o")
plt.xscale("log")
plt.ylim(-1, 1)

plt.xlabel("global reparameterization scale s")
plt.ylabel("correlation with test error")
plt.title("Global rescaling preserves correlation but changes sharpness units")
plt.grid(True, which="both")

fig_path = FIGURES / "23_correlation_by_scale.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 5. Save results

The results table includes both the original and reparameterized sharpness values.

The summary JSON records the main correlations.

In [ ]:
csv_path = RESULTS / "23_correlation_results.csv"
json_path = RESULTS / "23_correlation_summary.json"

models.to_csv(csv_path, index=False)

summary = {
    "n_models": int(n_models),
    "original_correlation_sharpness_test_error": float(corr_original),
    "reparameterized_correlation_sharpness_test_error": float(corr_reparam),
    "min_sharpness_ratio": float(models["sharpness_ratio"].min()),
    "max_sharpness_ratio": float(models["sharpness_ratio"].max()),
    "engineering_statement": "Correlation can survive observation, but causation must survive representation."
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)
print(summary)

## 6. Notebook takeaway

This synthetic experiment does not claim to reproduce the full empirical literature on flat minima.

It isolates a logic problem:

> If a measured quantity changes under function-preserving reparameterization, then that quantity is not automatically a coordinate-invariant causal explanation.

Flatness may remain useful as a diagnostic, predictor, or empirical correlate.

But causal explanation requires more.

**Engineering statement:** Correlation can survive observation. Causation must survive representation.

In [ ]:
output_files = [
    FIGURES / "23_original_correlation.png",
    FIGURES / "23_reparameterized_scatter.png",
    FIGURES / "23_correlation_by_scale.png",
    FIGURES / "23_before_after_correlation.png",
    RESULTS / "23_correlation_results.csv",
    RESULTS / "23_correlation_summary.json",
]

zip_path = DOWNLOADS / "23_correlation_vs_causation_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file_path in output_files:
        if file_path.exists():
            z.write(file_path, arcname=file_path.relative_to(ROOT))

print("Created:", zip_path)
print("\nIncluded files:")
for file_path in output_files:
    print("-", file_path.relative_to(ROOT), "exists=" + str(file_path.exists()))

## Download outputs

In Colab, run the next cell to download the ZIP directly.

In local Jupyter, open:

`downloads/23_correlation_vs_causation_outputs.zip`

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display

    display(FileLink(str(zip_path)))
    for file_path in output_files:
        if file_path.exists():
            display(FileLink(str(file_path)))